In [2]:
"""
statistical_tests.py

Statistical significance testing for the leakage-free 10-fold cross-validation
results, comparing any two ensemble configurations (Standard, OvO, OvR, HvL)
under either base classifier (Linear Perceptron or Naive Bayes), as reported
in the paper's Results and Discussion section.

DECISION PROCEDURE (per comparison):
  1. Compute the paired differences (a - b) across the 10 folds.
  2. Test these differences for normality using the Shapiro-Wilk test.
  3. If the differences are consistent with a normal distribution
     (Shapiro-Wilk p >= 0.05), the paired t-test is used as the primary,
     decisive test, since it is more statistically powerful under that
     assumption.
  4. If normality is rejected (Shapiro-Wilk p < 0.05), the Wilcoxon
     signed-rank test is used as the primary test instead, since it does
     not assume a particular distribution for the differences.
  5. Both the t-test and the Wilcoxon test are still computed and reported
     for every comparison, for transparency, but the "primary_test" and
     "primary_pvalue" columns record which test the decision procedure
     above selected and what its p-value was.
  6. Cohen's d for paired samples is reported alongside as an effect-size
     measure, independent of the normality decision.

Input files (expected in the same directory, or pass --input-dir):
  Results_{MODEL}_{ENSEMBLE}_LeakageFree_PerFold.xlsx
where MODEL is LP or NB, and ENSEMBLE is Standard, OvO, OvR, or HvL.
The script only needs the two files for the ensembles being compared
(--ensemble-a and --ensemble-b), not all four.

Each file must contain a "Metrics" sheet with columns:
  Dataset, Fold, Accuracy(%), WeightedF(%), MAE, RMSE, LOE, SOE, Time(s)
and one row per fold (0-9) plus a "MEAN" row per Dataset value.

Usage:
  # Linear Perceptron, Standard vs HvL (default)
  python statistical_tests.py --input-dir /path/to/xlsx/files

  # Naive Bayes, Standard vs HvL
  python statistical_tests.py --model NB --ensemble-a Standard --ensemble-b HvL --output nb_standard_vs_hvl.csv

  # Naive Bayes, OvR vs HvL
  python statistical_tests.py --model NB --ensemble-a OvR --ensemble-b HvL --output nb_ovr_vs_hvl.csv
"""

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon, shapiro

DATASETS = [
    "Core_Priority_withcross",
    "Core_Severity_withcross",
    "Firefox_Priority_withcross",
    "Firefox_Severity_withcross",
]
CONDITIONS = ["processed", "smote", "undersampled"]
METRICS = [("Accuracy(%)", "Accuracy"), ("SOE", "SOE")]

NORMALITY_ALPHA = 0.05


def load_fold_metrics(path: Path) -> pd.DataFrame:
    """Load the Metrics sheet and drop the MEAN summary rows, keeping folds 0-9."""
    df = pd.read_excel(path, sheet_name="Metrics")
    df = df[df["Fold"] != "MEAN"].copy()
    df["Fold"] = df["Fold"].astype(int)
    return df


def paired_cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    """Cohen's d for paired (repeated-measures) samples: mean(diff) / std(diff, ddof=1)."""
    diff = a - b
    sd = diff.std(ddof=1)
    if sd == 0:
        return float("nan")
    return diff.mean() / sd


def run_comparison(a_df: pd.DataFrame, b_df: pd.DataFrame, dataset: str, condition: str,
                    label_a: str = "Standard", label_b: str = "HvL"):
    """Run the normality check plus both tests for one dataset/condition, for both Accuracy and SOE."""
    key = f"{dataset}_{condition}"
    a_rows = a_df[a_df["Dataset"] == key].sort_values("Fold")
    b_rows = b_df[b_df["Dataset"] == key].sort_values("Fold")

    if len(a_rows) != 10 or len(b_rows) != 10:
        print(f"WARNING: expected 10 folds for {key}, got {label_a}={len(a_rows)}, {label_b}={len(b_rows)}",
              file=sys.stderr)
        return []

    results = []
    for col, label in METRICS:
        a = a_rows[col].to_numpy(dtype=float)
        b = b_rows[col].to_numpy(dtype=float)
        diff = a - b

        # Step 1: test the paired differences for normality
        try:
            shapiro_stat, shapiro_p = shapiro(diff)
        except ValueError:
            # Shapiro-Wilk requires at least 3 distinct values; degenerate case
            shapiro_stat, shapiro_p = np.nan, np.nan
        normal = (not np.isnan(shapiro_p)) and (shapiro_p >= NORMALITY_ALPHA)

        # Step 2: compute both tests regardless, for transparency
        t_stat, t_p = ttest_rel(a, b)
        try:
            w_stat, w_p = wilcoxon(a, b)
        except ValueError:
            # All differences are zero (only possible with degenerate data)
            w_stat, w_p = np.nan, np.nan

        # Step 3: pick the primary/decisive test based on the normality outcome
        if normal:
            primary_test = "t-test"
            primary_pvalue = t_p
        else:
            primary_test = "wilcoxon"
            primary_pvalue = w_p

        d = paired_cohens_d(a, b)

        results.append({
            "dataset": dataset.replace("_withcross", ""),
            "condition": condition,
            "metric": label,
            f"{label_a}_mean": round(a.mean(), 4),
            f"{label_b}_mean": round(b.mean(), 4),
            "mean_diff": round(diff.mean(), 4),
            "shapiro_stat": round(shapiro_stat, 4) if not np.isnan(shapiro_stat) else np.nan,
            "shapiro_pvalue": round(shapiro_p, 6) if not np.isnan(shapiro_p) else np.nan,
            "normal_distribution": normal,
            "primary_test": primary_test,
            "primary_pvalue": round(primary_pvalue, 6) if not np.isnan(primary_pvalue) else np.nan,
            "t_stat": round(t_stat, 4),
            "t_pvalue": round(t_p, 6),
            "wilcoxon_stat": w_stat,
            "wilcoxon_pvalue": round(w_p, 6) if not np.isnan(w_p) else np.nan,
            "cohens_d": round(d, 4),
        })
    return results


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--input-dir", type=Path, default=Path("."),
                         help="Directory containing the four per-fold Excel files")
    parser.add_argument("--output", type=Path, default=Path("results_significance.csv"),
                         help="Path to write the resulting CSV table")
    parser.add_argument("--model", type=str, default="LP", choices=["LP", "NB"],
                         help="Which classifier's per-fold files to load (LP or NB)")
    parser.add_argument("--ensemble-a", type=str, default="Standard",
                         choices=["Standard", "OvO", "OvR", "HvL"], help="First ensemble in the comparison")
    parser.add_argument("--ensemble-b", type=str, default="HvL",
                         choices=["Standard", "OvO", "OvR", "HvL"], help="Second ensemble in the comparison")
    args = parser.parse_args()

    a_path = args.input_dir / f"Results_{args.model}_{args.ensemble_a}_LeakageFree_PerFold.xlsx"
    b_path = args.input_dir / f"Results_{args.model}_{args.ensemble_b}_LeakageFree_PerFold.xlsx"

    a_df = load_fold_metrics(a_path)
    b_df = load_fold_metrics(b_path)

    all_results = []
    for dataset in DATASETS:
        for condition in CONDITIONS:
            all_results.extend(run_comparison(a_df, b_df, dataset, condition,
                                                label_a=args.ensemble_a, label_b=args.ensemble_b))

    out_df = pd.DataFrame(all_results)
    out_df.to_csv(args.output, index=False)

    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 20)
    print(out_df.to_string(index=False))

    n_normal = out_df["normal_distribution"].sum()
    n_total = len(out_df)
    print(f"\n{n_normal}/{n_total} comparisons had normally-distributed paired differences "
          f"(Shapiro-Wilk p >= {NORMALITY_ALPHA}) and used the paired t-test as the primary test; "
          f"the remaining {n_total - n_normal} used the Wilcoxon signed-rank test.")
    print(f"Saved to {args.output}")


#if __name__ == "__main__":
 #   main()

In [3]:
from pathlib import Path
import pandas as pd

input_dir = Path(".")

a_df = load_fold_metrics(input_dir / "Results_LP_Standard_LeakageFree_PerFold.xlsx")
b_df = load_fold_metrics(input_dir / "Results_LP_HvL_LeakageFree_PerFold.xlsx")

all_results = []
for dataset in DATASETS:
    for condition in CONDITIONS:
        all_results.extend(run_comparison(a_df, b_df, dataset, condition,
                                           label_a="Standard", label_b="HvL"))

out_df = pd.DataFrame(all_results)
out_df.to_csv("results_significance.csv", index=False)

n_normal = out_df["normal_distribution"].sum()
print(f"{n_normal}/{len(out_df)} karsilastirma normal dagilim gosterdi (t-test kullanildi)")
out_df

C:\Users\cando\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\cando\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


24/24 karsilastirma normal dagilim gosterdi (t-test kullanildi)


,dataset,condition,metric,Standard_mean,HvL_mean,mean_diff,shapiro_stat,shapiro_pvalue,normal_distribution,primary_test,primary_pvalue,t_stat,t_pvalue,wilcoxon_stat,wilcoxon_pvalue,cohens_d
0,Core_Priority,processed,Accuracy,69.5048,67.7418,1.7630,0.9736,0.922004,True,t-test,0.000000,20.3666,0.000000,0.0,0.001953,6.4405
1,Core_Priority,processed,SOE,0.8977,0.8176,0.0801,0.8759,0.116955,True,t-test,0.000000,22.4204,0.000000,0.0,0.001953,7.0900
2,Core_Priority,smote,Accuracy,65.5052,64.0642,1.4410,0.9648,0.838413,True,t-test,0.000011,8.7363,0.000011,0.0,0.001953,2.7627
3,Core_Priority,smote,SOE,1.1164,0.9488,0.1676,0.8929,0.182548,True,t-test,0.000000,27.7369,0.000000,0.0,0.001953,8.7712
4,Core_Priority,undersampled,Accuracy,64.8965,62.8948,2.0017,0.9474,0.637632,True,t-test,0.000020,8.1000,0.000020,0.0,0.001953,2.5615
5,Core_Priority,undersampled,SOE,1.1570,0.9910,0.1660,0.9598,0.783365,True,t-test,0.000000,20.0295,0.000000,0.0,0.001953,6.3339
6,Core_Severity,processed,Accuracy,73.8612,73.2194,0.6418,0.9848,0.985668,True,t-test,0.000213,5.9592,0.000213,0.0,0.001953,1.8845
7,Core_Severity,processed,SOE,0.4200,0.4072,0.0128,0.9097,0.278924,True,t-test,0.000012,8.5963,0.000012,0.0,0.001953,2.7184
8,Core_Severity,smote,Accuracy,67.7179,67.2430,0.4749,0.9197,0.354770,True,t-test,0.217724,1.3253,0.217724,16.0,0.275391,0.4191
9,Core_Severity,smote,SOE,0.5609,0.5200,0.0409,0.8837,0.143982,True,t-test,0.000130,6.3659,0.000130,0.0,0.001953,2.0131


In [4]:
a_df = load_fold_metrics(input_dir / "Results_NB_Standard_LeakageFree_PerFold.xlsx")
b_df = load_fold_metrics(input_dir / "Results_NB_HvL_LeakageFree_PerFold.xlsx")

all_results_nb1 = []
for dataset in DATASETS:
    for condition in CONDITIONS:
        all_results_nb1.extend(run_comparison(a_df, b_df, dataset, condition,
                                               label_a="Standard", label_b="HvL"))

out_df_nb1 = pd.DataFrame(all_results_nb1)
out_df_nb1.to_csv("nb_standard_vs_hvl.csv", index=False)

n_normal = out_df_nb1["normal_distribution"].sum()
print(f"{n_normal}/{len(out_df_nb1)} karsilastirma normal dagilim gosterdi (t-test kullanildi)")
out_df_nb1

C:\Users\cando\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\cando\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


22/24 karsilastirma normal dagilim gosterdi (t-test kullanildi)


,dataset,condition,metric,Standard_mean,HvL_mean,mean_diff,shapiro_stat,shapiro_pvalue,normal_distribution,primary_test,primary_pvalue,t_stat,t_pvalue,wilcoxon_stat,wilcoxon_pvalue,cohens_d
0,Core_Priority,processed,Accuracy,55.6660,51.4795,4.1865,0.8133,0.021041,False,wilcoxon,0.001953,24.5172,0.000000,0.0,0.001953,7.7530
1,Core_Priority,processed,SOE,1.6545,2.0006,-0.3462,0.9237,0.388998,True,t-test,0.000000,-54.6135,0.000000,0.0,0.001953,-17.2703
2,Core_Priority,smote,Accuracy,56.6637,49.1305,7.5332,0.9682,0.874099,True,t-test,0.000000,50.4371,0.000000,0.0,0.001953,15.9496
3,Core_Priority,smote,SOE,1.5666,2.0881,-0.5214,0.8988,0.212344,True,t-test,0.000000,-68.0292,0.000000,0.0,0.001953,-21.5127
4,Core_Priority,undersampled,Accuracy,54.7188,47.0581,7.6607,0.8999,0.218633,True,t-test,0.000000,57.4206,0.000000,0.0,0.001953,18.1580
5,Core_Priority,undersampled,SOE,1.7088,2.2741,-0.5653,0.8996,0.217104,True,t-test,0.000000,-63.4395,0.000000,0.0,0.001953,-20.0613
6,Core_Severity,processed,Accuracy,59.1667,59.2450,-0.0783,0.9716,0.904947,True,t-test,0.632635,-0.4947,0.632635,24.0,0.769531,-0.1565
7,Core_Severity,processed,SOE,0.8147,0.7920,0.0227,0.9459,0.620581,True,t-test,0.000194,6.0351,0.000194,0.0,0.001953,1.9085
8,Core_Severity,smote,Accuracy,60.9945,52.7449,8.2496,0.9691,0.882444,True,t-test,0.000000,32.2847,0.000000,0.0,0.001953,10.2093
9,Core_Severity,smote,SOE,0.7645,0.9859,-0.2214,0.9496,0.664065,True,t-test,0.000000,-38.2857,0.000000,0.0,0.001953,-12.1070


In [5]:
a_df = load_fold_metrics(input_dir / "Results_NB_OvR_LeakageFree_PerFold.xlsx")
b_df = load_fold_metrics(input_dir / "Results_NB_HvL_LeakageFree_PerFold.xlsx")

all_results_nb2 = []
for dataset in DATASETS:
    for condition in CONDITIONS:
        all_results_nb2.extend(run_comparison(a_df, b_df, dataset, condition,
                                               label_a="OvR", label_b="HvL"))

out_df_nb2 = pd.DataFrame(all_results_nb2)
out_df_nb2.to_csv("nb_ovr_vs_hvl.csv", index=False)

n_normal = out_df_nb2["normal_distribution"].sum()
print(f"{n_normal}/{len(out_df_nb2)} karsilastirma normal dagilim gosterdi (t-test kullanildi)")
out_df_nb2

C:\Users\cando\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\cando\anaconda3\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


19/24 karsilastirma normal dagilim gosterdi (t-test kullanildi)


,dataset,condition,metric,OvR_mean,HvL_mean,mean_diff,shapiro_stat,shapiro_pvalue,normal_distribution,primary_test,primary_pvalue,t_stat,t_pvalue,wilcoxon_stat,wilcoxon_pvalue,cohens_d
0,Core_Priority,processed,Accuracy,58.9621,51.4795,7.4827,0.9767,0.944895,True,t-test,0.000000,37.4428,0.000000,0.0,0.001953,11.8405
1,Core_Priority,processed,SOE,1.5336,2.0006,-0.4670,0.7834,0.009111,False,wilcoxon,0.001953,-30.7025,0.000000,0.0,0.001953,-9.7090
2,Core_Priority,smote,Accuracy,57.1739,49.1305,8.0434,0.9443,0.601325,True,t-test,0.000000,77.5541,0.000000,0.0,0.001953,24.5248
3,Core_Priority,smote,SOE,1.6542,2.0881,-0.4339,0.9132,0.303776,True,t-test,0.000000,-54.7731,0.000000,0.0,0.001953,-17.3208
4,Core_Priority,undersampled,Accuracy,52.9533,47.0581,5.8952,0.8297,0.033162,False,wilcoxon,0.001953,27.6652,0.000000,0.0,0.001953,8.7485
5,Core_Priority,undersampled,SOE,1.8852,2.2741,-0.3889,0.8996,0.216922,True,t-test,0.000000,-31.1955,0.000000,0.0,0.001953,-9.8649
6,Core_Severity,processed,Accuracy,61.8930,59.2450,2.6480,0.8422,0.046837,False,wilcoxon,0.001953,18.5163,0.000000,0.0,0.001953,5.8554
7,Core_Severity,processed,SOE,0.7315,0.7920,-0.0605,0.8489,0.056407,True,t-test,0.000000,-13.9015,0.000000,0.0,0.001953,-4.3960
8,Core_Severity,smote,Accuracy,57.3915,52.7449,4.6466,0.9197,0.354082,True,t-test,0.000000,18.6790,0.000000,0.0,0.001953,5.9068
9,Core_Severity,smote,SOE,0.8446,0.9859,-0.1413,0.9634,0.823534,True,t-test,0.000000,-34.7749,0.000000,0.0,0.001953,-10.9968
